In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('../data/processed/model_ready_data.csv')

In [3]:
drop_cols_reg = ['Date', 'Revenue', 'ROI', 'Profit_Flag']
X = df.drop(columns=drop_cols_reg)
y = df['Revenue']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)

(133029, 31) (33258, 31)


In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

In [8]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    results[name] = {'R2': r2, 'MAE': mae, 'RMSE': rmse}
    print(f"{name} → R²: {r2:.4f}, MAE: {mae:.2f}, RMSE: {rmse:.2f}")

Linear Regression → R²: 0.7749, MAE: 150238.77, RMSE: 232106.89
Random Forest → R²: 0.7684, MAE: 152485.03, RMSE: 235458.14
XGBoost → R²: 0.7698, MAE: 151143.20, RMSE: 234759.72


In [7]:
from sklearn.ensemble import GradientBoostingRegressor, ExtraTreesRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [ ]:

models_round2 = {
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42),
    'Extra Trees': ExtraTreesRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    'LightGBM': LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=7, random_state=42),
    'CatBoost': CatBoostRegressor(n_estimators=500, learning_rate=0.05, depth=7, random_state=42, verbose=0)
}

for name, model in models_round2.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    print(f"{name} → R²: {r2:.4f}, MAE: {mae:.2f}, RMSE: {rmse:.2f}")

Gradient Boosting → R²: 0.7722, MAE: 150667.42, RMSE: 233500.01
Extra Trees → R²: 0.7684, MAE: 152432.64, RMSE: 235476.39
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011681 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1858
[LightGBM] [Info] Number of data points in the train set: 133029, number of used features: 31
[LightGBM] [Info] Start training from score 511542.473446
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
LightGBM → R²: 0.7685, MAE: 151129.89, RMSE: 235402.91
CatBoost → R²: 0.7732, MAE: 150605.91, RMSE: 233004.12


In [9]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'n_estimators': [200, 400, 600, 800],
    'max_depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

search = RandomizedSearchCV(
    XGBRegressor(random_state=42),
    param_distributions=param_grid,
    n_iter=25,
    scoring='r2',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)
print("Best params:", search.best_params_)
print("Best R² (CV):", search.best_score_)

best_model = search.best_estimator_
preds = best_model.predict(X_test) # type: ignore
print("Test R²:", r2_score(y_test, preds))

Fitting 3 folds for each of 25 candidates, totalling 75 fits


Best params: {'subsample': 0.8, 'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.01, 'colsample_bytree': 1.0}
Best R² (CV): 0.7727595369021097
Test R²: 0.7732733488082886


In [10]:
import pickle

with open('../model/regression_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('../model/regression_features.pkl', 'wb') as f:
    pickle.dump(list(X_train.columns), f)